# Startup Script

In [1]:
# CELL 1: Setup (run this first every new session)
import os
import torch
import numpy as np
import time

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone repo if not exists
if not os.path.exists('/content/Medusa'):
    !git clone https://github.com/ZakiNabeel/Medusa.git

# Install dependencies
!pip install transformers==4.36.0 -q
!pip install pyopencl -q

# Add to path
import sys
sys.path.insert(0, '/content/Medusa')

# Import everything
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from medusa_opencl_verify import (
    build_dynamic_tree,
    run_medusa_verify_opencl,
    trace_longest_path,
    build_opencl_context
)

print("Setup complete. Ready to run pipeline.")

Mounted at /content/drive
Cloning into 'Medusa'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (217/217), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 372 (delta 157), reused 137 (delta 125), pack-reused 155 (from 1)
Receiving objects: 100% (372/372), 4.93 MiB | 26.41 MiB/s, done.
Resolving deltas: 100% (203/203), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 61.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.0 which is incompatible.
   ━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Setup complete. Ready to run pipeline.


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
import torch
print(torch.cuda.is_available())        # must print True
print(torch.cuda.get_device_name(0))    # should say Tesla T4

True
Tesla T4


## clone and setup repo

In [3]:
!git clone https://github.com/ZakiNabeel/Medusa.git
%cd Medusa

fatal: destination path 'Medusa' already exists and is not an empty directory.
/content/Medusa


In [4]:
import os
for root, dirs, files in os.walk('.'):
    # skip hidden folders and git internals
    dirs[:] = [d for d in dirs if not d.startswith('.')]
    level = root.replace('.', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    for file in files:
        print(f'{indent}  {file}')

./
  deepspeed.json
  pyproject.toml
  README.md
  CITATION.cff
  simple_gradio_interface.py
  ROADMAP.md
  medusa_opencl_verify.py
  test_opencl.py
  .gitignore
  mock_baseline.py
  LICENSE
  create_data.py
  scripts/
    train_vicuna_33b_8bit.sh
    train_vicuna_7b.sh
  notebooks/
    medusa_inference_explained.ipynb
    medusa_introduction.ipynb
    medusa_configuration_explained.ipynb
  medusa/
    __init__.py
    hf_utils.py
    inference/
      __init__.py
      cli.py
    model/
      modeling_llama_kv.py
      medusa_model_legacy.py
      medusa_choices.py
      medusa_model.py
      kv_cache.py
      utils_legacy.py
      modeling_mistral_kv.py
      modeling_llama_kv_legacy.py
      medusa_model_new.py
      __init__.py
      utils.py
    train/
      train_legacy.py
      __init__.py
    eval/
      gen_results.py
      README.md
      heads_accuracy.py
  data_generation/
    convert_to_sharegpt.py
    README.md
    generate.py
  llm_judge/
    gen_judgement.py
    README.md

In [5]:
!pip install -e . -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.0/807.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.2/79.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 7.8 MB/s eta 0:00:00
  Building editable for medusa-llm (pyproject.toml) ... done


In [6]:
!pip install pyopencl -q

In [7]:
import pyopencl as cl
platforms = cl.get_platforms()
for p in platforms:
    print(p.name)
    for d in p.get_devices():
        print('  ', d.name, d.type)

NVIDIA CUDA
   Tesla T4 4


## mount!!

In [9]:
from google.colab import drive
drive.mount('/content/drive')

# create a project folder
!mkdir -p /content/drive/MyDrive/medusa_project_2


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Model Integration Side


## Setup

In [11]:
!pwd
!ls

/content/Medusa
assets		 medusa			  README.md
CITATION.cff	 medusa_llm.egg-info	  ROADMAP.md
create_data.py	 medusa_opencl_verify.py  scripts
data_generation  mock_baseline.py	  simple_gradio_interface.py
deepspeed.json	 notebooks		  test_opencl.py
LICENSE		 __pycache__
llm_judge	 pyproject.toml


In [12]:
# install the old transformers, new ones not compatible w old medusa
!pip install transformers==4.36.0 -q

#move into repo and reinstall
%cd /content/Medusa
!pip install -e . -q

/content/Medusa
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for medusa-llm (pyproject.toml) ... done


In [13]:
import torch
import torch.nn as nn
from transformers import GPT2LMHeadModel, GPT2Tokenizer, GPT2Config

# Load GPT-2
model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = model.cuda()
model.eval()

# Create simple Medusa heads
class SimpleMedusaHeads(nn.Module):
    def __init__(self, hidden_dim, vocab_size, num_heads=3):
        super().__init__()
        self.heads = nn.ModuleList([
            nn.Linear(hidden_dim, vocab_size, bias=False)
            for _ in range(num_heads)
        ])

    def forward(self, hidden_states):
        last_hidden = hidden_states[:, -1, :]
        return [head(last_hidden) for head in self.heads]

hidden_dim = model.config.n_embd  # 768 for GPT-2
vocab_size = model.config.vocab_size  # 50257
medusa_heads = SimpleMedusaHeads(hidden_dim, vocab_size, num_heads=3).cuda()

# Get hidden states from GPT-2
def get_hidden_and_logits(model, input_ids):
    outputs = model(input_ids, output_hidden_states=True)
    hidden_states = outputs.hidden_states[-1]  # Last layer
    base_logits = outputs.logits
    return hidden_states, base_logits

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Baseline decoder

In [20]:
import time

def baseline_decode(prompt, max_new_tokens=50):
    input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()
    generated = input_ids.clone()

    start = time.time()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(generated)
            next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=-1)
    elapsed = time.time() - start

    tps = max_new_tokens / elapsed
    text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return text, tps

# Test it
text, tps = baseline_decode("The future of artificial intelligence is", max_new_tokens=50)
print(text)
print(f"\nBaseline speed: {tps:.2f} tokens/second")

BASELINE_TPS = tps
print(f"\n>>> BASELINE: {BASELINE_TPS:.2f} tokens/second <<<")

The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer science at the University of California, Berkeley. "But we're not sure what the future will look

Baseline speed: 34.13 tokens/second

>>> BASELINE: 34.13 tokens/second <<<


#### it was 2.66, 20.2, 28.2, 30.1, 21.99, 47.21, 34.3 = (26.38)avg tokens/second

*   List item
*   List item



## Extract real logits from forward pass

In [21]:
prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()

with torch.no_grad():
    hidden_states, base_logits = get_hidden_and_logits(model, input_ids)
    head_outputs = medusa_heads(hidden_states)

#convert to numpy for the pipeline
head_logits_list = [h[0].cpu().numpy() for h in head_outputs]

print(f"Number of heads: {len(head_logits_list)}")
print(f"Head 0 logits shape: {head_logits_list[0].shape}")
print(f"Head 0 logits sample (first 5): {head_logits_list[0][:5]}")
print(f"Head 0 max value: {head_logits_list[0].max():.4f}")

Number of heads: 3
Head 0 logits shape: (50257,)
Head 0 logits sample (first 5): [ 1.6951623 -3.9890509  5.490915  -1.7217057  5.5986094]
Head 0 max value: 16.7503


## import code, build tree

In [22]:
import sys
sys.path.insert(0, '/content/Medusa')

from medusa_opencl_verify import (
    build_dynamic_tree,
    run_medusa_verify_opencl,
    trace_longest_path,
    build_opencl_context
)

# Build the tree
conf = 0.01 #start w a very low threshold
tree = build_dynamic_tree(
    head_logits_list,
    confidence_threshold=conf,
    max_nodes=256
)

print(f"Tree nodes: {tree.num_nodes}")
print(f"Pruned branches: {tree.pruned_branches}")
print(f"Active levels: {tree.num_levels}")

# If tree.num_nodes is 1 (just root, nothing survived), lower threshold to 0.001
if tree.num_nodes <= 1:
    print("WARNING: No nodes survived. Lowering threshold to 0.001")
    tree = build_dynamic_tree(head_logits_list, confidence_threshold=0.001, max_nodes=256)
    print(f"New tree nodes: {tree.num_nodes}")

Tree nodes: 256
Pruned branches: 150744
Active levels: 3


In [23]:
# see what tree size you get without the cap just bc
tree_no_cap = build_dynamic_tree(
    head_logits_list,
    confidence_threshold=0.01,
    max_nodes=10000
)

print(f"Tree without cap: {tree_no_cap.num_nodes} nodes") #it's 865 nodes BTW

Tree without cap: 778 nodes


## Run OpenCL Kernel

In [25]:
def get_base_model_verification_logits(model, input_ids, tree):
    """
    For each node, get what the BASE MODEL predicts as the next token
    This is the verification step — the heads propose, the base model disposes.
    """
    import torch

    base_verify_logits = np.zeros_like(tree.node_logits)

    with torch.no_grad():
        for node_idx in range(1, tree.num_nodes):
            # Walk up the tree to get the path to this node
            path_tokens = []
            cur = node_idx
            while cur != 0:  # 0 is root
                path_tokens.append(tree.candidates[cur])
                cur = tree.parent_indices[cur]
            path_tokens.reverse()  # Now from root to parent of this node

            # Build the sequence: original prompt + path tokens so far
            if len(path_tokens) == 0:
                # This node is directly attached to root
                seq = input_ids
            else:
                # Exclude the current node itself — base model predicts what comes NEXT
                extra = torch.tensor(path_tokens, dtype=torch.long).unsqueeze(0).cuda()
                seq = torch.cat([input_ids, extra], dim=-1)

            outputs = model(seq)
            # The base model's prediction for the next token after seq
            base_verify_logits[node_idx] = outputs.logits[0, -1, :].cpu().numpy()

    return base_verify_logits

print("Getting base model verification logits...")
real_verify_logits = get_base_model_verification_logits(model, input_ids, tree)

# use these logits for verification
verify_candidates = tree.candidates[1:]
verify_logits = real_verify_logits[1:]
verify_parents = tree.parent_indices[1:]

verify_parents_reindexed = np.where(
    verify_parents == 0, -1, verify_parents - 1
).astype(np.int32)
ctx, queue, program = build_opencl_context()
# Run OpenCL verification on base model logits
is_match, elapsed = run_medusa_verify_opencl(
    verify_candidates, verify_logits,
    ctx=ctx, queue=queue, program=program
)

best_node, accepted_length = trace_longest_path(is_match, verify_parents_reindexed)

print(f"Kernel time: {elapsed*1000:.3f} ms")
print(f"Accepted tokens: {accepted_length}")
print(f"Matches: {is_match.sum()} out of {len(is_match)} nodes")

Getting base model verification logits...
[OpenCL] Platform  : NVIDIA CUDA
[OpenCL] Device    : Tesla T4
[OpenCL] Max WG    : 1024
[OpenCL] LDS total : 48 KB
[OpenCL] LDS used  : 2048 bytes  (4.2% of limit per WG)
Kernel time: 0.242 ms
Accepted tokens: 0
Matches: 0 out of 255 nodes


/usr/local/lib/python3.12/dist-packages/pyopencl/__init__.py:570: CompilerWarning: Non-empty compiler output encountered. Set the environment variable PYOPENCL_COMPILER_OUTPUT=1 to see more.
  lambda: self._prg.build(options_bytes, devices),


## decode accepted tokens

In [26]:
if accepted_length > 0:
    # Walk path to get accepted token sequence
    path_tokens = []
    cur = best_node
    while cur != -1:
        path_tokens.append(int(verify_candidates[cur]))
        parent = int(verify_parents_reindexed[cur])
        cur = parent if parent >= 0 else -1
    path_tokens.reverse()

    accepted_ids = torch.tensor(path_tokens).unsqueeze(0).cuda()
    new_ids = torch.cat([input_ids, accepted_ids], dim=-1)
    print("Output with Medusa (verified):")
    print(tokenizer.decode(new_ids[0], skip_special_tokens=True))

    # Now the correctness check should pass
    print(f"\nBaseline would produce token IDs: {greedy_next_tokens(model, input_ids, accepted_length)}")
    print(f"Medusa accepted token IDs: {path_tokens}")
    if path_tokens == greedy_next_tokens(model, input_ids, accepted_length):
        print("✓ CORRECT! Medusa accepted the same tokens as baseline.")
    else:
        print("Mismatch — still debugging, but heads are random so length may be 0")
else:
    print("No tokens accepted. With random heads this is expected.")
    # Fall back to baseline greedy
    next_token = base_logits[0, -1, :].argmax().item()
    new_ids = torch.cat([input_ids, torch.tensor([[next_token]]).cuda()], dim=-1)
    print(tokenizer.decode(new_ids[0], skip_special_tokens=True))

No tokens accepted. With random heads this is expected.
The future of artificial intelligence is uncertain


## verify correctness against baseline

In [27]:
def greedy_next_tokens(model, input_ids, n):
    """What would pure greedy decoding produce for the next n tokens?"""
    generated = input_ids.clone()
    tokens = []
    with torch.no_grad():
        for _ in range(n):
            out = model(generated)
            next_tok = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            tokens.append(next_tok.item())
            generated = torch.cat([generated, next_tok], dim=-1)
    return tokens

# Get what baseline would have produced
expected = greedy_next_tokens(model, input_ids, n=3)
print(f"Baseline would produce: {expected}")
print(f"Baseline tokens decoded: {[tokenizer.decode([t]) for t in expected]}")
print(f"Medusa accepted: {path_tokens if accepted_length > 0 else 'none'}")

# The first `accepted_length` tokens in path_tokens should match
# the first `accepted_length` tokens in expected
if accepted_length > 0:
    if path_tokens == expected[:accepted_length]:
        print("Correctness check passed")
    else:
        print("MISMATCH! Medusa accepted different tokens than baseline.")
        print(f"  Expected first {accepted_length}: {expected[:accepted_length]}")
        print(f"  Got: {path_tokens}")
else:
    print("No speculative tokens were accepted (expected with random heads)")
    print("This is fine — the heads are untrained and guessing randomly.")

Baseline would produce: [8627, 13, 198]
Baseline tokens decoded: [' uncertain', '.', '\n']
Medusa accepted: none
No speculative tokens were accepted (expected with random heads)
This is fine — the heads are untrained and guessing randomly.


## save model state + notebook

In [28]:
# Save the model state and notebook
!cp /content/Medusa/medusa_opencl_verify.py /content/drive/MyDrive/medusa_project_2/

# Also save any trained weights later
# torch.save(medusa_heads.state_dict(), '/content/drive/MyDrive/medusa_project/medusa_heads_untrained.pt')

# Model Training

## ***(1)Packed Data Loaded***

In [29]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import torch
import math

# ── Config ────────────────────────────────────────────────────────────────────
CHUNK_SIZE   = 512    # tokens per training chunk (fits T4 @ batch=8 comfortably)
BATCH_SIZE   = 8      # physical batch size
GRAD_ACCUM   = 4      # accumulate over 4 mini-batches → effective batch = 32
NUM_EPOCHS   = 2      # full passes over the packed dataset
WARMUP_RATIO = 0.10   # first 10% of steps used for LR warm-up
EVAL_EVERY   = 300    # print accuracy metrics every N *optimizer* steps
LOG_EVERY    = 50     # print loss every N optimizer steps
SAVE_PATH    = "/content/drive/MyDrive/medusa_project/trained_heads.pt"
# ─────────────────────────────────────────────────────────────────────────────

print("Loading WikiText-2 …")
raw_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# ── 1. Tokenise the entire split (no truncation, no padding) ─────────────────
def tokenize_fn(examples):
    texts = [t for t in examples["text"] if t.strip()]
    return tokenizer(texts, truncation=False, padding=False, add_special_tokens=False)

tokenized = raw_dataset.map(
    tokenize_fn, batched=True, remove_columns=["text"],
    desc="Tokenising"
)

# ── 2. Concatenate all token IDs into one long 1-D list ──────────────────────
all_ids = []
for sample in tokenized:
    all_ids.extend(sample["input_ids"])

print(f"Total tokens after concatenation: {len(all_ids):,}")

# ── 3. Slice into non-overlapping CHUNK_SIZE chunks ──────────────────────────
n_chunks = (len(all_ids) - 1) // CHUNK_SIZE
all_ids   = all_ids[:n_chunks * CHUNK_SIZE + 1]
token_tensor = torch.tensor(all_ids, dtype=torch.long)

class PackedDataset(Dataset):
    """Fixed-length contiguous chunks; zero padding, zero waste."""
    def __init__(self, tokens: torch.Tensor, chunk_size: int):
        self.tokens     = tokens
        self.chunk_size = chunk_size
        self.n_chunks   = (len(tokens) - 1) // chunk_size

    def __len__(self):
        return self.n_chunks

    def __getitem__(self, idx):
        start = idx * self.chunk_size
        x = self.tokens[start : start + self.chunk_size]
        y = self.tokens[start + 1 : start + self.chunk_size + 1]
        return x, y

packed_ds     = PackedDataset(token_tensor, CHUNK_SIZE)
train_loader  = DataLoader(packed_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)

steps_per_epoch = len(train_loader) // GRAD_ACCUM
total_steps     = steps_per_epoch * NUM_EPOCHS
warmup_steps    = int(total_steps * WARMUP_RATIO)

print(f"Chunks  : {len(packed_ds):,}  |  Batches/epoch: {len(train_loader):,}")
print(f"Optimizer steps/epoch : {steps_per_epoch:,}")
print(f"Total optimizer steps : {total_steps:,}  (warmup: {warmup_steps})")

Loading WikiText-2 …


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Tokenising:   0%|          | 0/36718 [00:00<?, ? examples/s]

Total tokens after concatenation: 2,391,884
Chunks  : 4,671  |  Batches/epoch: 583
Optimizer steps/epoch : 145
Total optimizer steps : 290  (warmup: 29)


## ***Optimizer, Scheduler, Evaluation***

In [31]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

optimizer = AdamW(medusa_heads.parameters(), lr=3e-4, weight_decay=1e-2)

def cosine_with_warmup(step: int) -> float:
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = LambdaLR(optimizer, lr_lambda=cosine_with_warmup)

criterion     = nn.CrossEntropyLoss()
lambda_weights = [0.8 ** (k + 1) for k in range(3)]
hidden_dim     = model.config.n_embd

@torch.no_grad()
def evaluate_heads(model, medusa_heads, eval_loader, n_batches: int = 30):
    medusa_heads.eval()
    correct_top1  = [0] * 3
    correct_top10 = [0] * 3
    total_tokens  = [0] * 3

    for i, (x, y) in enumerate(eval_loader):
        if i >= n_batches: break
        x, y = x.cuda(), y.cuda()

        with torch.no_grad():
            out = model(x, output_hidden_states=True)
            hidden = out.hidden_states[-1]

        for k in range(3):
            shift = k + 1
            if hidden.shape[1] <= shift: continue

            h_in   = hidden[:, :-shift, :].reshape(-1, hidden_dim)
            logits = medusa_heads.heads[k](h_in)

            tgt = y[:, k : y.shape[1] - shift + k + 1]
            if tgt.shape[1] != h_in.shape[0] // x.shape[0]:
                tgt_len = h_in.shape[0] // x.shape[0]
                tgt = y[:, k : k + tgt_len]
            tgt_flat = tgt.reshape(-1)

            if tgt_flat.shape[0] != logits.shape[0]: continue

            pred1 = logits.argmax(dim=-1)
            correct_top1[k]  += (pred1 == tgt_flat).sum().item()

            top10 = logits.topk(10, dim=-1).indices
            hit10 = (top10 == tgt_flat.unsqueeze(1)).any(dim=1)
            correct_top10[k] += hit10.sum().item()
            total_tokens[k]  += tgt_flat.shape[0]

    medusa_heads.train()
    results = {}
    for k in range(3):
        if total_tokens[k] > 0:
            results[k] = {
                "top1":  100.0 * correct_top1[k]  / total_tokens[k],
                "top10": 100.0 * correct_top10[k] / total_tokens[k],
            }
        else:
            results[k] = {"top1": 0.0, "top10": 0.0}
    return results

## **Training Loop** *italicized text*

In [32]:
import time

model.eval()
model.requires_grad_(False)
medusa_heads.train()

global_opt_step  = 0
running_loss     = 0.0
best_head0_top1  = 0.0
accum_step       = 0

print("=" * 70)
print(f"  MEDUSA HEAD TRAINING  —  {NUM_EPOCHS} epochs, {total_steps} opt-steps")
print("=" * 70)

optimizer.zero_grad()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()
    print(f"\n{'─'*70}\n  EPOCH {epoch + 1} / {NUM_EPOCHS}\n{'─'*70}")

    for micro_step, (x, y) in enumerate(train_loader):
        x, y = x.cuda(), y.cuda()

        with torch.no_grad():
            out    = model(x, output_hidden_states=True)
            hidden = out.hidden_states[-1]

        total_loss     = torch.tensor(0.0, device="cuda")
        active_heads   = 0

        for k in range(3):
            shift = k + 1
            if hidden.shape[1] <= shift: continue

            h_in   = hidden[:, :-shift, :].reshape(-1, hidden_dim)
            logits = medusa_heads.heads[k](h_in)

            tgt_len  = h_in.shape[0] // x.shape[0]
            tgt      = y[:, k : k + tgt_len].reshape(-1)

            if tgt.shape[0] != logits.shape[0]: continue

            loss          = criterion(logits, tgt)
            total_loss   += lambda_weights[k] * loss
            active_heads += 1

        if active_heads == 0: continue

        scaled_loss = total_loss / (active_heads * GRAD_ACCUM)
        scaled_loss.backward()

        running_loss += total_loss.item() / active_heads
        accum_step   += 1

        if accum_step % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(medusa_heads.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
            global_opt_step += 1

            if global_opt_step % LOG_EVERY == 0:
                avg_loss = running_loss / LOG_EVERY
                lr_now   = scheduler.get_last_lr()[0]
                running_loss = 0.0
                print(f"  step {global_opt_step:>5} / {total_steps}  |  loss {avg_loss:.4f}  |  lr {lr_now:.2e}")

            if global_opt_step % EVAL_EVERY == 0:
                acc = evaluate_heads(model, medusa_heads, train_loader, n_batches=30)
                print(f"\n  ── Accuracy @ step {global_opt_step} ────────────────────────────")
                for k in range(3):
                    print(f"     Head {k}: Top-1 = {acc[k]['top1']:5.2f}%  Top-10 = {acc[k]['top10']:5.2f}%")

                if acc[0]["top1"] > best_head0_top1:
                    best_head0_top1 = acc[0]["top1"]
                    torch.save(
                        {"model_state_dict": medusa_heads.state_dict()},
                        SAVE_PATH,
                    )
                    print(f"  ✓ New best Top-1 ({best_head0_top1:.2f}%) → checkpoint saved\n")

# Final Evaluation
medusa_heads.eval()
print("\n" + "=" * 70 + "\n  TRAINING COMPLETE\n" + "=" * 70)
final_acc = evaluate_heads(model, medusa_heads, train_loader, n_batches=100)
for k in range(3):
    print(f"  Head {k} final → Top-1 = {final_acc[k]['top1']:5.2f}%  Top-10 = {final_acc[k]['top10']:5.2f}%")

  MEDUSA HEAD TRAINING  —  2 epochs, 290 opt-steps

──────────────────────────────────────────────────────────────────────
  EPOCH 1 / 2
──────────────────────────────────────────────────────────────────────
  step    50 / 290  |  loss 41.8744  |  lr 2.95e-04
  step   100 / 290  |  loss 23.8122  |  lr 2.48e-04

──────────────────────────────────────────────────────────────────────
  EPOCH 2 / 2
──────────────────────────────────────────────────────────────────────
  step   150 / 290  |  loss 18.2105  |  lr 1.67e-04
  step   200 / 290  |  loss 16.4778  |  lr 7.97e-05
  step   250 / 290  |  loss 16.0881  |  lr 1.71e-05

  TRAINING COMPLETE
  Head 0 final → Top-1 = 28.30%  Top-10 = 52.42%
  Head 1 final → Top-1 = 12.62%  Top-10 = 34.68%
  Head 2 final → Top-1 =  7.85%  Top-10 = 28.83%


## ***Check***

In [34]:
import numpy as np

prompt    = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors="pt").cuda()

with torch.no_grad():
    hidden_states, base_logits = get_hidden_and_logits(model, input_ids)
    head_outputs = medusa_heads(hidden_states)

head_logits_list = [h[0].cpu().numpy() for h in head_outputs]

print("Top-5 predictions per head after training:")
for idx, logits in enumerate(head_logits_list):
    # Added .copy() here to fix the negative stride issue
    top_k      = np.argsort(logits)[-5:][::-1].copy()
    top_tokens = [tokenizer.decode([t]) for t in top_k]
    top_probs  = torch.softmax(torch.tensor(logits), dim=-1)[top_k].numpy()
    print(f"  Head {idx}: {list(zip(top_tokens, top_probs.round(3)))}")

Top-5 predictions per head after training:
  Head 0: [(' a', np.float32(0.062)), (' not', np.float32(0.048)), (' in', np.float32(0.03)), (' an', np.float32(0.016)), (' being', np.float32(0.009))]
  Head 1: [(' a', np.float32(0.032)), (' to', np.float32(0.03)), (' in', np.float32(0.03)), (' by', np.float32(0.027)), (' the', np.float32(0.019))]
  Head 2: [(' the', np.float32(0.063)), (',', np.float32(0.026)), (' to', np.float32(0.021)), (' of', np.float32(0.015)), (' by', np.float32(0.015))]


### quick test

In [35]:
# Quick test after training
prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()

with torch.no_grad():
    hidden_states, base_logits = get_hidden_and_logits(model, input_ids)
    head_outputs = medusa_heads(hidden_states)

head_logits_list = [h[0].cpu().numpy() for h in head_outputs]

# Check top predictions for each head
for idx, logits in enumerate(head_logits_list):
    top_k = np.argsort(logits)[-5:][::-1]
    top_tokens = [tokenizer.decode([t]) for t in top_k]
    print(f"Head {idx} top 5 predictions: {top_tokens}")

Head 0 top 5 predictions: [' a', ' not', ' in', ' an', ' being']
Head 1 top 5 predictions: [' a', ' to', ' in', ' by', ' the']
Head 2 top 5 predictions: [' the', ',', ' to', ' of', ' by']


## top k approach like in paper instead of confidence threshold

In [36]:
import sys
sys.path.insert(0, '/content/Medusa')
import time
from medusa_opencl_verify import (
    build_dynamic_tree,
    run_medusa_verify_opencl,
    trace_longest_path,
    build_opencl_context
)

def medusa_generate(
    prompt,
    model,
    medusa_heads,
    tokenizer,
    max_new_tokens=50,
    k=10,  # top-k candidates per head
    max_tree_nodes=256,
    verbose=False
):
    """
    Full Medusa speculative generation with trained heads.
    """
    input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()
    generated = input_ids.clone()

    # Setup OpenCL once
    ctx, queue, program = build_opencl_context()

    start_time = time.time()
    total_tokens_generated = 0

    tokens_per_step = []
    accepted_per_step = []

    with torch.no_grad():
        while generated.shape[1] - input_ids.shape[1] < max_new_tokens:
            # Get hidden states and head predictions
            outputs = model(generated, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            base_logits = outputs.logits

            # Get head outputs
            head_outputs = medusa_heads(hidden_states)
            head_logits_list = [h[0].cpu().numpy() for h in head_outputs]

            # Build tree using top-k
            topk_tokens = []
            for logits in head_logits_list:
                top_k_indices = np.argsort(logits)[-k:][::-1]
                topk_tokens.append(top_k_indices.tolist())

            # Build candidate tree
            candidates = [-1]
            parent_indices = [-1]
            node_level = [0]

            for level, tokens in enumerate(topk_tokens, 1):
                current_nodes = [i for i in range(len(candidates)) if node_level[i] == level - 1]
                for parent_node in current_nodes:
                    for token in tokens:
                        if len(candidates) >= max_tree_nodes:
                            break
                        candidates.append(token)
                        parent_indices.append(parent_node)
                        node_level.append(level)
                    if len(candidates) >= max_tree_nodes:
                        break
                if len(candidates) >= max_tree_nodes:
                    break

            if len(candidates) <= 1:
                # No tree built, fallback
                next_token = base_logits[0, -1, :].argmax().item()
                generated = torch.cat([generated, torch.tensor([[next_token]]).cuda()], dim=-1)
                total_tokens_generated += 1
                tokens_per_step.append(1)
                accepted_per_step.append(0)
                continue

            # Build node_logits
            vocab_size = head_logits_list[0].shape[0]
            node_logits = np.zeros((len(candidates), vocab_size), dtype=np.float32)
            for i in range(1, len(candidates)):
                level = node_level[i]
                node_logits[i] = head_logits_list[level - 1]

            # Get base model verification (simplified — just first few nodes for speed)
            # For full speed, you'd use tree attention, but this works for demo
            verify_candidates = np.array(candidates[1:], dtype=np.int32)
            verify_parents = np.array(parent_indices[1:], dtype=np.int32)
            verify_parents_reindexed = np.where(verify_parents == 0, -1, verify_parents - 1).astype(np.int32)

            # Get base model predictions for candidate positions (simplified)
            verify_logits = np.zeros((len(verify_candidates), vocab_size), dtype=np.float32)
            for node_idx in range(len(verify_candidates)):
                path_tokens = []
                cur = node_idx + 1
                while cur != 0:
                    path_tokens.append(candidates[cur])
                    cur = parent_indices[cur]
                path_tokens.reverse()

                if len(path_tokens) <= 1:
                    seq = generated
                else:
                    extra = torch.tensor(path_tokens[:-1], dtype=torch.long).unsqueeze(0).cuda()
                    seq = torch.cat([generated, extra], dim=-1)

                outputs = model(seq)
                verify_logits[node_idx] = outputs.logits[0, -1, :].cpu().numpy()

            # Run OpenCL verification
            is_match, _ = run_medusa_verify_opencl(
                verify_candidates, verify_logits,
                ctx=ctx, queue=queue, program=program
            )

            # Find longest accepted path
            valid_lengths = np.zeros(len(is_match))
            for i in range(len(is_match)):
                if is_match[i] == 1:
                    parent = verify_parents_reindexed[i]
                    if parent == -1:
                        valid_lengths[i] = 1
                    elif valid_lengths[parent] > 0:
                        valid_lengths[i] = valid_lengths[parent] + 1

            best_idx = np.argmax(valid_lengths)
            accepted_len = int(valid_lengths[best_idx])

            if accepted_len > 0:
                # Build accepted token sequence
                path_tokens = []
                cur = best_idx
                while cur != -1:
                    path_tokens.append(int(verify_candidates[cur]))
                    parent = int(verify_parents_reindexed[cur])
                    cur = parent if parent >= 0 else -1
                path_tokens.reverse()

                # Append accepted tokens
                accepted_ids = torch.tensor(path_tokens, dtype=torch.long).unsqueeze(0).cuda()
                generated = torch.cat([generated, accepted_ids], dim=-1)
                total_tokens_generated += accepted_len
                accepted_per_step.append(accepted_len)
                tokens_per_step.append(accepted_len)
            else:
                # No accepted tokens, take greedy
                next_token = base_logits[0, -1, :].argmax().item()
                generated = torch.cat([generated, torch.tensor([[next_token]]).cuda()], dim=-1)
                total_tokens_generated += 1
                tokens_per_step.append(1)
                accepted_per_step.append(0)

            if verbose and len(tokens_per_step) % 5 == 0:
                print(f"Step {len(tokens_per_step)}: Generated {total_tokens_generated} tokens so far")

    elapsed = time.time() - start_time
    tps = total_tokens_generated / elapsed

    output_text = tokenizer.decode(generated[0], skip_special_tokens=True)

    return {
        'text': output_text,
        'tokens_per_second': tps,
        'total_tokens': total_tokens_generated,
        'time_seconds': elapsed,
        'tokens_per_step': tokens_per_step,
        'accepted_per_step': accepted_per_step,
        'acceptance_rate': sum(accepted_per_step) / (sum(accepted_per_step) + sum(1 for a in accepted_per_step if a == 0)) if accepted_per_step else 0
    }

# Run it!
print("Running Medusa generation with trained heads...")
result = medusa_generate(
    "The future of artificial intelligence is",
    model,
    medusa_heads,
    tokenizer,
    max_new_tokens=30,
    k=10,
    verbose=True
)

print(f"\n{'='*60}")
print(f"Medusa Output:")
print(result['text'])
print(f"\n{'='*60}")
print(f"Performance:")
print(f"  Total tokens: {result['total_tokens']}")
print(f"  Time: {result['time_seconds']:.2f} seconds")
print(f"  Tokens/second: {result['tokens_per_second']:.2f}")
print(f"  Acceptance rate: {result['acceptance_rate']:.2%}")
print(f"{'='*60}")

# Compare with baseline
print("\nRunning baseline for comparison...")
baseline_text, baseline_tps = baseline_decode("The future of artificial intelligence is", max_new_tokens=30)
print(f"Baseline tokens/second: {baseline_tps:.2f}")
print(f"Medusa tokens/second: {result['tokens_per_second']:.2f}")
print(f"Speedup: {result['tokens_per_second'] / baseline_tps:.2f}x")

Running Medusa generation with trained heads...
[OpenCL] Platform  : NVIDIA CUDA
[OpenCL] Device    : Tesla T4
[OpenCL] Max WG    : 1024
[OpenCL] LDS total : 48 KB
[OpenCL] LDS used  : 2048 bytes  (4.2% of limit per WG)
Step 5: Generated 5 tokens so far
Step 10: Generated 11 tokens so far
Step 15: Generated 16 tokens so far
Step 20: Generated 21 tokens so far
Step 25: Generated 27 tokens so far

Medusa Output:
The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer

Performance:
  Total tokens: 30
  Time: 119.95 seconds
  Tokens/second: 0.25
  Acceptance rate: 33.33%

Running baseline for comparison...
Baseline tokens/second: 84.52
Medusa tokens/second: 0.25
Speedup: 0.00x


## new version: tree attention batching
because the bottleneck is now the python for loop

In [39]:
import sys
sys.path.insert(0, '/content/Medusa')
import time
import numpy as np
import torch
from medusa_opencl_verify import (
    run_medusa_verify_opencl,
    trace_longest_path,
    build_opencl_context
)

def build_tree_topk(head_logits_list, k=10, max_nodes=256):
    """Build candidate tree using top-k sampling"""
    topk_tokens = []
    for logits in head_logits_list:
        top_k_indices = np.argsort(logits)[-k:][::-1]
        topk_tokens.append(top_k_indices.tolist())

    candidates = [-1]
    parent_indices = [-1]
    node_level = [0]

    for level, tokens in enumerate(topk_tokens, 1):
        current_nodes = [i for i in range(len(candidates)) if node_level[i] == level - 1]
        for parent_node in current_nodes:
            for token in tokens:
                if len(candidates) >= max_nodes:
                    break
                candidates.append(token)
                parent_indices.append(parent_node)
                node_level.append(level)
            if len(candidates) >= max_nodes:
                break
        if len(candidates) >= max_nodes:
            break

    # Build node_logits
    vocab_size = head_logits_list[0].shape[0]
    node_logits = np.zeros((len(candidates), vocab_size), dtype=np.float32)
    for i in range(1, len(candidates)):
        level = node_level[i]
        node_logits[i] = head_logits_list[level - 1]

    return {
        'candidates': np.array(candidates, dtype=np.int32),
        'parent_indices': np.array(parent_indices, dtype=np.int32),
        'node_logits': node_logits,
        'num_nodes': len(candidates),
        'node_level': node_level
    }

def verify_candidates_batched(model, input_ids, tree):
    """
    Optimized verification — groups candidates by their prefix
    to minimize base model forward passes.

    Instead of 155 forward passes, this does ~30-50 passes.
    """
    candidates = tree['candidates']
    parent_indices = tree['parent_indices']
    num_nodes = tree['num_nodes']
    vocab_size = tree['node_logits'].shape[1]

    # Group nodes by their prefix (path to parent)
    prefix_map = {}

    for node_idx in range(1, num_nodes):
        # Get path tokens to parent (excluding current node)
        path_tokens = []
        cur = parent_indices[node_idx]
        while cur > 0:
            path_tokens.append(candidates[cur])
            cur = parent_indices[cur]
        path_tokens.reverse()

        prefix_key = tuple(path_tokens)
        if prefix_key not in prefix_map:
            prefix_map[prefix_key] = []
        prefix_map[prefix_key].append(node_idx)

    # Initialize verify logits
    verify_logits = np.zeros((num_nodes, vocab_size), dtype=np.float32)

    # Process each unique prefix with one model call
    with torch.no_grad():
        for prefix_key, node_indices in prefix_map.items():
            # Build sequence once per unique prefix
            if len(prefix_key) == 0:
                seq = input_ids
            else:
                prefix_tensor = torch.tensor(prefix_key, dtype=torch.long).unsqueeze(0).cuda()
                seq = torch.cat([input_ids, prefix_tensor], dim=-1)

            # Single forward pass for all nodes sharing this prefix
            outputs = model(seq)
            base_logits = outputs.logits[0, -1, :].cpu().numpy()

            # Assign same logits to all nodes in this group
            for node_idx in node_indices:
                verify_logits[node_idx] = base_logits

    return verify_logits

def medusa_generate_optimized(
    prompt,
    model,
    medusa_heads,
    tokenizer,
    max_new_tokens=50,
    k=10,
    max_tree_nodes=256,
    verbose=False
):
    """
    Optimized Medusa generation with batched verification.
    """
    input_ids = tokenizer.encode(prompt, return_tensors='pt').cuda()
    generated = input_ids.clone()

    # Setup OpenCL once
    ctx, queue, program = build_opencl_context()

    start_time = time.time()
    total_tokens_generated = 0
    step_times = []
    verification_times = []
    kernel_times = []

    with torch.no_grad():
        while generated.shape[1] - input_ids.shape[1] < max_new_tokens:
            step_start = time.time()

            # Get hidden states and head predictions
            outputs = model(generated, output_hidden_states=True)
            hidden_states = outputs.hidden_states[-1]
            base_logits = outputs.logits

            # Get head outputs
            head_outputs = medusa_heads(hidden_states)
            head_logits_list = [h[0].cpu().numpy() for h in head_outputs]

            # Build tree
            tree = build_tree_topk(head_logits_list, k=k, max_nodes=max_tree_nodes)

            if tree['num_nodes'] <= 1:
                # No tree built, fallback to greedy
                next_token = base_logits[0, -1, :].argmax().item()
                generated = torch.cat([generated, torch.tensor([[next_token]]).cuda()], dim=-1)
                total_tokens_generated += 1
                step_times.append(time.time() - step_start)
                continue

            # Batched verification — this is the optimization
            verify_start = time.time()
            verify_logits = verify_candidates_batched(model, input_ids, tree)
            verification_times.append(time.time() - verify_start)

            # Prepare for OpenCL kernel
            verify_candidates = tree['candidates'][1:]
            verify_logits_nodes = verify_logits[1:]
            verify_parents = tree['parent_indices'][1:]

            verify_parents_reindexed = np.where(
                verify_parents == 0, -1, verify_parents - 1
            ).astype(np.int32)

            # Run OpenCL kernel
            kernel_start = time.time()
            is_match, _ = run_medusa_verify_opencl(
                verify_candidates, verify_logits_nodes,
                ctx=ctx, queue=queue, program=program
            )
            kernel_times.append(time.time() - kernel_start)

            # Find longest accepted path
            valid_lengths = np.zeros(len(is_match))
            for i in range(len(is_match)):
                if is_match[i] == 1:
                    parent = verify_parents_reindexed[i]
                    if parent == -1:
                        valid_lengths[i] = 1
                    elif valid_lengths[parent] > 0:
                        valid_lengths[i] = valid_lengths[parent] + 1

            best_idx = np.argmax(valid_lengths)
            accepted_len = int(valid_lengths[best_idx])

            if accepted_len > 0:
                # Build accepted token sequence
                path_tokens = []
                cur = best_idx
                while cur != -1:
                    path_tokens.append(int(verify_candidates[cur]))
                    parent = int(verify_parents_reindexed[cur])
                    cur = parent if parent >= 0 else -1
                path_tokens.reverse()

                # Append accepted tokens
                accepted_ids = torch.tensor(path_tokens, dtype=torch.long).unsqueeze(0).cuda()
                generated = torch.cat([generated, accepted_ids], dim=-1)
                total_tokens_generated += accepted_len
            else:
                # No accepted tokens, take greedy
                next_token = base_logits[0, -1, :].argmax().item()
                generated = torch.cat([generated, torch.tensor([[next_token]]).cuda()], dim=-1)
                total_tokens_generated += 1

            step_times.append(time.time() - step_start)

            if verbose and len(step_times) % 10 == 0:
                avg_verify = np.mean(verification_times[-10:]) if verification_times else 0
                avg_kernel = np.mean(kernel_times[-10:]) if kernel_times else 0
                print(f"Step {len(step_times)}: {total_tokens_generated} tokens, "
                      f"verify={avg_verify*1000:.1f}ms, kernel={avg_kernel*1000:.3f}ms")

    elapsed = time.time() - start_time
    tps = total_tokens_generated / elapsed

    # Statistics
    avg_step_time = np.mean(step_times)
    avg_verify_time = np.mean(verification_times) if verification_times else 0
    avg_kernel_time = np.mean(kernel_times) if kernel_times else 0

    output_text = tokenizer.decode(generated[0], skip_special_tokens=True)

    return {
        'text': output_text,
        'tokens_per_second': tps,
        'total_tokens': total_tokens_generated,
        'time_seconds': elapsed,
        'avg_step_time_ms': avg_step_time * 1000,
        'avg_verification_time_ms': avg_verify_time * 1000,
        'avg_kernel_time_ms': avg_kernel_time * 1000,
        'num_steps': len(step_times),
        'acceptance_rate': (total_tokens_generated - len(step_times)) / total_tokens_generated if total_tokens_generated > 0 else 0
    }

# Run optimized Medusa
print("="*60)
print("Running OPTIMIZED Medusa generation with batched verification...")
print("="*60)

result = medusa_generate_optimized(
    "The future of artificial intelligence is",
    model,
    medusa_heads,
    tokenizer,
    max_new_tokens=50,
    k=10,
    verbose=True
)

print(f"\n{'='*60}")
print(f"Medusa Output:")
print(result['text'])
print(f"\n{'='*60}")
print(f"Performance Summary:")
print(f"  Total tokens generated: {result['total_tokens']}")
print(f"  Time: {result['time_seconds']:.2f} seconds")
print(f"  Tokens/second: {result['tokens_per_second']:.2f}")
print(f"  Acceptance rate: {result['acceptance_rate']:.2%}")
print(f"\nBreakdown per step:")
print(f"  Average step time: {result['avg_step_time_ms']:.1f} ms")
print(f"  Average verification (batched): {result['avg_verification_time_ms']:.1f} ms")
print(f"  Average OpenCL kernel: {result['avg_kernel_time_ms']:.3f} ms")
print(f"{'='*60}")

# Compare with baseline
print("\nRunning baseline for comparison...")
baseline_text, baseline_tps = baseline_decode("The future of artificial intelligence is", max_new_tokens=50)
print(f"\n{'='*60}")
print(f"FINAL COMPARISON:")
print(f"  Baseline: {baseline_tps:.2f} tokens/second")
print(f"  Medusa (optimized): {result['tokens_per_second']:.2f} tokens/second")
print(f"  Speedup: {result['tokens_per_second'] / baseline_tps:.2f}x")
print(f"{'='*60}")

Running OPTIMIZED Medusa generation with batched verification...
[OpenCL] Platform  : NVIDIA CUDA
[OpenCL] Device    : Tesla T4
[OpenCL] Max WG    : 1024
[OpenCL] LDS total : 48 KB
[OpenCL] LDS used  : 2048 bytes  (4.2% of limit per WG)
Step 10: 10 tokens, verify=445.2ms, kernel=55.027ms
Step 20: 20 tokens, verify=360.2ms, kernel=47.633ms
Step 30: 30 tokens, verify=374.4ms, kernel=49.724ms
Step 40: 40 tokens, verify=414.0ms, kernel=52.130ms
Step 50: 50 tokens, verify=373.0ms, kernel=50.275ms

Medusa Output:
The future of artificial intelligence is uncertain.

"We're not sure what the future will look like," said Dr. Michael S. Schoenfeld, a professor of computer science at the University of California, Berkeley. "But we're not sure what the future will look

Performance Summary:
  Total tokens generated: 50
  Time: 24.99 seconds
  Tokens/second: 2.00
  Acceptance rate: 0.00%

Breakdown per step:
  Average step time: 499.7 ms
  Average verification (batched): 393.3 ms
  Average OpenCL k

## save trained heads to drive

In [43]:
import torch

# Save trained heads with a clean checkpoint
checkpoint = {
    'model_state_dict': medusa_heads.state_dict(),
    'config': {
        'num_heads': 3,
        'hidden_dim': hidden_dim,
        'vocab_size': vocab_size,
    }
}

# Save to Drive
torch.save(checkpoint, '/content/drive/MyDrive/medusa_project/trained_heads.pt')
print("Saved final trained heads checkpoint to Drive")

Saved final trained heads checkpoint to Drive


# full medusa pipeline with entropy calculation (NEW)
changed to use trained heads

In [48]:
%%writefile medusa_pipeline.py
"""
medusa_pipeline.py  –  End-to-End Speculative Decoding Pipeline (Naive Version)
================================================================
"""

import argparse
import time
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer

# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 0 – GPT-2 + Medusa Heads
# ══════════════════════════════════════════════════════════════════════════════

class GPT2Medusa(nn.Module):
    def __init__(self, base_model_name: str = "gpt2", num_heads: int = 3):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        self.base_model = AutoModelForCausalLM.from_pretrained(base_model_name)

        # Move to GPU
        self.base_model = self.base_model.cuda()

        self.hidden_size = self.base_model.config.n_embd
        self.vocab_size = self.base_model.config.vocab_size
        self.num_heads = num_heads

        self.medusa_heads = nn.ModuleList([
            nn.Linear(self.hidden_size, self.vocab_size, bias=False)
            for _ in range(num_heads)
        ])

    def forward(self, text_prompt: str):
        inputs = self.tokenizer(text_prompt, return_tensors="pt")
        inputs = {k: v.cuda() for k, v in inputs.items()}
        outputs = self.base_model(**inputs, output_hidden_states=True)

        base_logits = outputs.logits[0, -1, :].detach().cpu().numpy()
        last_hidden = outputs.hidden_states[-1][0, -1, :]

        medusa_logits = []
        for head in self.medusa_heads:
            medusa_logits.append(head(last_hidden).detach().cpu().numpy())

        return base_logits, medusa_logits


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 1 – Shannon Entropy + Adaptive Threshold
# ══════════════════════════════════════════════════════════════════════════════

def _softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


def shannon_entropy(logits: np.ndarray) -> float:
    probs = _softmax(logits.astype(np.float32))
    mask = probs > 0.0
    return float(-np.sum(probs[mask] * np.log(probs[mask])))


def entropy_to_threshold(entropy: float, max_entropy: float, low_thr: float = 0.02, high_thr: float = 0.15) -> float:
    """
    CORRECT: Low entropy (confident) → low threshold → more candidates
             High entropy (uncertain) → high threshold → fewer candidates (prune)
    """
    if max_entropy <= 0:
        return high_thr
    ratio = min(entropy / max_entropy, 1.0)
    return low_thr + ratio * (high_thr - low_thr)


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 2 – Dynamic Candidate Tree Builder
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class DynamicTreeOutput:
    candidates: np.ndarray
    parent_indices: np.ndarray
    head_node_logits: np.ndarray
    num_nodes: int
    num_levels: int
    pruned_branches: int


def build_dynamic_tree(medusa_head_logits: List[np.ndarray], confidence_threshold: float = 0.15, max_nodes: int = 256) -> DynamicTreeOutput:
    vocab_size = medusa_head_logits[0].shape[0]
    num_heads = len(medusa_head_logits)
    pruned_total = 0

    head_probs = [_softmax(h.astype(np.float32)) for h in medusa_head_logits]

    candidates_list = [-1]
    parent_list = [-1]
    node_level = [0]
    node_head_idx = [0]

    frontier = [0]
    num_levels_active = 0

    for head_idx in range(num_heads):
        if not frontier or len(candidates_list) >= max_nodes:
            break

        probs = head_probs[head_idx]
        passing_mask = probs > confidence_threshold
        passing_ids = np.where(passing_mask)[0]
        pruned_total += int((~passing_mask).sum())

        if len(passing_ids) == 0:
            break

        next_frontier = []

        for parent_node_idx in frontier:
            for token_id in passing_ids:
                new_node_idx = len(candidates_list)
                if new_node_idx >= max_nodes:
                    break
                candidates_list.append(int(token_id))
                parent_list.append(parent_node_idx)
                node_level.append(head_idx + 1)
                node_head_idx.append(head_idx)
                next_frontier.append(new_node_idx)
            if len(candidates_list) >= max_nodes:
                break

        frontier = next_frontier
        num_levels_active += 1

    num_nodes = len(candidates_list)
    candidates = np.array(candidates_list, dtype=np.int32)
    parent_indices = np.array(parent_list, dtype=np.int32)

    head_node_logits = np.zeros((num_nodes, vocab_size), dtype=np.float32)
    for i in range(1, num_nodes):
        head_node_logits[i] = medusa_head_logits[node_head_idx[i]]

    return DynamicTreeOutput(
        candidates=candidates,
        parent_indices=parent_indices,
        head_node_logits=head_node_logits,
        num_nodes=num_nodes,
        num_levels=num_levels_active,
        pruned_branches=pruned_total,
    )


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 3 – Base Model Verification (BATCHED)
# ══════════════════════════════════════════════════════════════════════════════

def get_base_model_verification_logits(model, input_ids, tree):
    vocab_size = model.vocab_size
    num_verify = tree.num_nodes - 1

    sequences = []
    lengths = []

    for verify_idx in range(num_verify):
        tree_node_idx = verify_idx + 1

        path_tokens = []
        cur = tree_node_idx
        while cur != 0:
            path_tokens.append(int(tree.candidates[cur]))
            cur = int(tree.parent_indices[cur])
        path_tokens.reverse()

        prefix = path_tokens[:-1]

        if len(prefix) == 0:
            seq = input_ids[0]
        else:
            extra = torch.tensor(prefix, dtype=torch.long).cuda()
            seq = torch.cat([input_ids[0], extra], dim=0)

        sequences.append(seq)
        lengths.append(seq.shape[0])

    max_len = max(lengths)
    padded = torch.zeros(num_verify, max_len, dtype=torch.long).cuda()
    for i, seq in enumerate(sequences):
        padded[i, :seq.shape[0]] = seq

    with torch.no_grad():
        outputs = model.base_model(padded)

    verify_logits = np.zeros((num_verify, vocab_size), dtype=np.float32)
    for i, length in enumerate(lengths):
        verify_logits[i] = outputs.logits[i, length - 1, :].cpu().numpy()

    return verify_logits


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 4 – OpenCL Hardware-Accelerated Verification
# ══════════════════════════════════════════════════════════════════════════════

WG_SIZE = 256

KERNEL_SOURCE = r"""
#define WG_SIZE 256

__kernel __attribute__((reqd_work_group_size(WG_SIZE, 1, 1)))
void medusa_verify_local(
    __global const float* restrict logits,
    __global const int*   restrict drafts,
    __global       int*   restrict is_match,
    const int vocab_size
) {
    const int lid = get_local_id(0);
    const int node_id = get_group_id(0);
    const __global float* row = logits + (long)node_id * vocab_size;

    float my_val = -INFINITY;
    int my_idx = 0;
    for (int v = lid; v < vocab_size; v += WG_SIZE) {
        float val = row[v];
        if (val > my_val) { my_val = val; my_idx = v; }
    }

    __local float local_vals[WG_SIZE];
    __local int local_idxs[WG_SIZE];
    local_vals[lid] = my_val;
    local_idxs[lid] = my_idx;
    barrier(CLK_LOCAL_MEM_FENCE);

    for (int stride = WG_SIZE >> 1; stride > 0; stride >>= 1) {
        if (lid < stride) {
            float ov = local_vals[lid + stride];
            int oi = local_idxs[lid + stride];
            if (ov > local_vals[lid] ||
               (ov == local_vals[lid] && oi < local_idxs[lid])) {
                local_vals[lid] = ov;
                local_idxs[lid] = oi;
            }
        }
        barrier(CLK_LOCAL_MEM_FENCE);
    }

    if (lid == 0) {
        is_match[node_id] = (local_idxs[0] == drafts[node_id]) ? 1 : 0;
    }
}
"""


def build_opencl_context(platform_idx: int = 0, device_idx: int = 0):
    import pyopencl as cl

    platform = cl.get_platforms()[platform_idx]
    device = platform.get_devices()[device_idx]
    ctx = cl.Context([device])
    queue = cl.CommandQueue(ctx, properties=cl.command_queue_properties.PROFILING_ENABLE)
    program = cl.Program(ctx, KERNEL_SOURCE).build(options="-cl-fast-relaxed-math -cl-mad-enable")

    lds_bytes = WG_SIZE * (4 + 4)
    print(f"  [OpenCL] Platform: {platform.name}")
    print(f"  [OpenCL] Device: {device.name}")
    print(f"  [OpenCL] Max WG: {device.max_work_group_size}")
    print(f"  [OpenCL] LDS total: {device.local_mem_size // 1024} KB")
    print(f"  [OpenCL] LDS/WG: {lds_bytes} bytes ({lds_bytes / device.local_mem_size * 100:.1f}%)")

    return ctx, queue, program


def run_medusa_verify_opencl(drafts_np: np.ndarray, logits_np: np.ndarray, ctx=None, queue=None, program=None) -> Tuple[np.ndarray, float]:
    import pyopencl as cl

    if ctx is None:
        ctx, queue, program = build_opencl_context()

    num_nodes, vocab_size = logits_np.shape
    drafts_np = np.ascontiguousarray(drafts_np, dtype=np.int32)
    logits_np = np.ascontiguousarray(logits_np, dtype=np.float32)

    mf = cl.mem_flags
    logits_buf = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=logits_np)
    drafts_buf = cl.Buffer(ctx, mf.READ_ONLY | mf.COPY_HOST_PTR, hostbuf=drafts_np)
    match_buf = cl.Buffer(ctx, mf.WRITE_ONLY, size=drafts_np.nbytes)

    kernel = cl.Kernel(program, "medusa_verify_local")
    event = kernel(queue, (num_nodes * WG_SIZE,), (WG_SIZE,), logits_buf, drafts_buf, match_buf, np.int32(vocab_size))
    event.wait()

    elapsed = (event.profile.end - event.profile.start) * 1e-9
    is_match = np.empty(num_nodes, dtype=np.int32)
    cl.enqueue_copy(queue, is_match, match_buf)
    queue.finish()

    return is_match, elapsed


def run_medusa_verify_numpy(drafts_np: np.ndarray, logits_np: np.ndarray) -> Tuple[np.ndarray, float]:
    t0 = time.perf_counter()
    argmaxes = np.argmax(logits_np, axis=1)
    is_match = (argmaxes == drafts_np).astype(np.int32)
    return is_match, time.perf_counter() - t0


# ══════════════════════════════════════════════════════════════════════════════
#  STAGE 5 – Path Tracing & Correctness Check
# ══════════════════════════════════════════════════════════════════════════════

def trace_longest_path(is_match: np.ndarray, parent_indices: np.ndarray) -> Tuple[int, int]:
    valid_path_lengths = np.zeros_like(is_match)

    for i in range(len(is_match)):
        if is_match[i] == 1:
            parent = parent_indices[i]
            if parent == -1:
                valid_path_lengths[i] = 1
            elif valid_path_lengths[parent] > 0:
                valid_path_lengths[i] = valid_path_lengths[parent] + 1

    best_node = int(np.argmax(valid_path_lengths))
    path_length = int(valid_path_lengths[best_node])
    return best_node, path_length


# ══════════════════════════════════════════════════════════════════════════════
#  FULL PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(prompt: str = "The capital of France is", num_heads: int = 3, low_thr: float = 0.02, high_thr: float = 0.15, max_nodes: int = 256) -> None:
    SEP = "═" * 68
    sep2 = "─" * 68

    print(f"\n{SEP}")
    print("  MEDUSA SPECULATIVE DECODING PIPELINE (FINAL CORRECTED)")
    print(SEP)
    print(f"  Prompt: \"{prompt}\"")
    print(f"  Heads: {num_heads} | max_nodes: {max_nodes}")
    print(f"  Threshold range: [{low_thr}, {high_thr}] (low=expand, high=prune)")
    print(sep2)

    # STAGE 1 – Neural Inference
    print("\n  STAGE 1 — Neural Inference")
    print(sep2)

    print("  Loading GPT-2 + Medusa heads...")
    t_inf_start = time.perf_counter()
    model = GPT2Medusa(num_heads=num_heads)

    # Load trained heads
    checkpoint = torch.load('/content/drive/MyDrive/medusa_project/trained_heads.pt')
    state_dict = checkpoint['model_state_dict']
    new_state_dict = {key.replace('heads.', ''): value for key, value in state_dict.items()}
    model.medusa_heads.load_state_dict(new_state_dict)
    model.medusa_heads.cuda()
    model.medusa_heads.eval()
    print("  Loaded trained Medusa heads!")

    # Get input IDs for later
    input_ids = model.tokenizer(prompt, return_tensors="pt")["input_ids"].cuda()

    print("  Running forward pass...")
    base_logits, medusa_logits = model(prompt)
    t_inference = time.perf_counter() - t_inf_start

    vocab_size = base_logits.shape[0]
    max_entropy = float(np.log(vocab_size))

    print(f"  Vocab size: {vocab_size:,}")
    print(f"  Inference time: {t_inference * 1e3:.1f} ms")
    print(f"  Base argmax token: {int(np.argmax(base_logits))} (\"{model.tokenizer.decode([int(np.argmax(base_logits))])}\")")
    print()

    # Per-head entropy
    entropies = []
    thresholds = []
    print(f"  {'Head':>4}  {'Entropy (nats)':>16}  {'Max H':>8}  {'Threshold':>10}  {'Direction'}")
    print(f"  {'----':>4}  {'---------------':>16}  {'-------':>8}  {'----------':>10}  {'----------'}")

    for i, h_logits in enumerate(medusa_logits):
        H = shannon_entropy(h_logits)
        thr = entropy_to_threshold(H, max_entropy, low_thr, high_thr)
        entropies.append(H)
        thresholds.append(thr)
        # FIX 1: Correct direction label
        direction = "← wide (expand)" if H < 0.6 * max_entropy else "→ narrow (prune)"
        print(f"  {i:>4}  {H:>16.4f}  {max_entropy:>8.4f}  {thr:>10.4f}  {direction}")

    # STAGE 2 – Tree Construction
    print(f"\n{sep2}")
    print("  STAGE 2 — Entropy-Based Tree Construction")
    print(sep2)

    mean_threshold = float(np.mean(thresholds))
    print(f"  Mean adaptive threshold: {mean_threshold:.5f}")

    t_tree_start = time.perf_counter()
    tree = build_dynamic_tree(medusa_logits, confidence_threshold=mean_threshold, max_nodes=max_nodes)
    t_tree = time.perf_counter() - t_tree_start

    print(f"  Tree build time: {t_tree * 1e3:.2f} ms")
    print(f"  Total nodes (incl. root): {tree.num_nodes}")
    print(f"  Active levels: {tree.num_levels} (of {num_heads} heads)")
    print(f"  Branches pruned: {tree.pruned_branches}")

    # Prepare verification arrays
    verify_candidates = tree.candidates[1:]
    verify_parents_raw = tree.parent_indices[1:]
    verify_parents = np.where(verify_parents_raw == 0, -1, verify_parents_raw - 1).astype(np.int32)
    num_verify = len(verify_candidates)

    if num_verify == 0:
        print("\n  Tree is empty — no candidates to verify.")
        return

    # STAGE 3 – Base Model Verification (BATCHED)
    print(f"\n{sep2}")
    print(f"  STAGE 3 — Base Model Verification ({num_verify} candidate nodes)")
    print(sep2)

    print("  Running batched base model verification (ONE forward pass)...")
    t_base_start = time.perf_counter()
    real_verify_logits = get_base_model_verification_logits(model, input_ids, tree)
    t_base = time.perf_counter() - t_base_start
    print(f"  Batched verification time: {t_base * 1e3:.1f} ms")

    # STAGE 4 – GPU Verification
    print(f"\n{sep2}")
    print(f"  STAGE 4 — GPU Verification ({num_verify} candidate nodes)")
    print(sep2)

    is_match_np, t_numpy = run_medusa_verify_numpy(verify_candidates, real_verify_logits)
    print(f"  [NumPy] time = {t_numpy * 1e3:.4f} ms, matches = {is_match_np.sum()}")

    use_opencl = False
    is_match = is_match_np
    t_gpu = t_numpy

    try:
        ctx, queue, program = build_opencl_context()
        is_match_cl, t_cl = run_medusa_verify_opencl(verify_candidates, real_verify_logits, ctx=ctx, queue=queue, program=program)
        print(f"  [OpenCL] time = {t_cl * 1e3:.4f} ms, matches = {is_match_cl.sum()}")

        if np.array_equal(is_match_np, is_match_cl):
            print("  ✓ NumPy and OpenCL match")
        is_match = is_match_cl
        t_gpu = t_cl
        use_opencl = True
    except Exception as exc:
        print(f"  [OpenCL] Error: {exc}")

    # STAGE 5 – Logic Resolution & Correctness Check
    print(f"\n{sep2}")
    print("  STAGE 5 — Logic Resolution")
    print(sep2)

    best_node, accepted_length = trace_longest_path(is_match, verify_parents)

    if accepted_length > 0:
        path_tokens = []
        cur = best_node
        while cur != -1:
            path_tokens.append(int(verify_candidates[cur]))
            parent = int(verify_parents[cur])
            cur = parent if parent >= 0 else -1
        path_tokens.reverse()
        accepted_text = model.tokenizer.decode(path_tokens)
        print(f"  ✓ Accepted {accepted_length} speculative token(s)")
        print(f"  Decoded text: \"{accepted_text}\"")

        # FIX 4: Correctness check against greedy baseline
        greedy_tokens = []
        check_ids = input_ids.clone()
        with torch.no_grad():
            for _ in range(accepted_length):
                out = model.base_model(check_ids)
                next_tok = out.logits[0, -1, :].argmax().item()
                greedy_tokens.append(next_tok)
                check_ids = torch.cat([check_ids, torch.tensor([[next_tok]]).cuda()], dim=-1)

        if path_tokens == greedy_tokens:
            print(f"  ✓ Correctness verified: accepted tokens match greedy baseline")
        else:
            print(f"  ✗ CORRECTNESS FAILURE: Expected {greedy_tokens}, Got {path_tokens}")
    else:
        print("  No speculative tokens accepted")

    # FIX 3: Accurate performance summary
    print()
    print(f"  ─── Performance Summary ───────────────────────────────────────")
    print(f"  Token efficiency this step: {accepted_length + 1:.1f}×")
    print(f"  (baseline: 1 token per forward pass, speculation: {accepted_length + 1} tokens)")
    if use_opencl and t_numpy > 0 and t_gpu > 0:
        print(f"  OpenCL verification speedup: {t_numpy / t_gpu:.2f}× over NumPy")
    print(f"  Batched base model verification: {t_base * 1e3:.1f} ms for {num_verify} candidates")

    print(f"\n{SEP}")
    print("  Pipeline complete.")
    print(SEP)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prompt", type=str, default="The future of artificial intelligence is")
    parser.add_argument("--heads", type=int, default=3)
    parser.add_argument("--low-thr", type=float, default=0.001)
    parser.add_argument("--high-thr", type=float, default=0.01)
    parser.add_argument("--max-nodes", type=int, default=256)
    args = parser.parse_args()

    run_pipeline(prompt=args.prompt, num_heads=args.heads, low_thr=args.low_thr, high_thr=args.high_thr, max_nodes=args.max_nodes)

Overwriting medusa_pipeline.py


In [53]:
!python medusa_pipeline.py --prompt "The future of artificial intelligence is" --heads 3 --low-thr 0.001 --high-thr 0.01

/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(

════════════════════════════════════════════════════════════════════
  MEDUSA SPECULATIVE DECODING PIPELINE (FINAL CORRECTED)
════════════════════════════════════════════════════════════════════
  Prompt: "The future of artificial intelligence is"
  Heads: 3 | max_nodes: 256
  Threshold range: [0.001, 0.01] (low=expand, high=prune)
────────────────────────────────────────────────────────────────────

  STAGE 1 — Neural Inference
────────────────────────────────────────────────────────────────────
  Loading GPT-2 + Medusa heads...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you wan

In [ ]:
# Copy to Drive
!cp /content/Medusa/medusa_pipeline.py /content/drive/MyDrive/medusa_project/medusa_pipeline.py